# Partea a treia. Transformer baseline
- la nivel de text, titlu, propozitie, summary. (?)

In [8]:
import pandas as pd
import re
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import torch.nn as nn
from transformers import AutoModel
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report

In [2]:
# FUNCTII UTILE le definim sus
def drop_columns(df, columns_to_drop):
    return df.drop(columns=columns_to_drop)

def clean_text(text):
    if not isinstance(text, str):
        return ""
        
    text = text.replace("\n", " ")
    text = text.replace('""', '"')
    text = re.sub(r"\s+", " ", text)
        
    return text.strip()

In [3]:
# citim
top20_ro_train_text = pd.read_csv('final_stilo/top20_ro_train.csv')
top20_ro_val_text = pd.read_csv('final_stilo/top20_ro_val.csv')
top20_ro_test_text = pd.read_csv('final_stilo/top20_ro_test.csv')

In [4]:
# PAS 1: Label encoding autori
# folosim data loader pentru bert (ca la pytorch)
# autorii trebuie encodati pentru ca bert nu lucreaza cu stringuri ca etichete
l_encoder = LabelEncoder()
l_encoder.fit(top20_ro_train_text['author'])

y_train_t20ro = l_encoder.transform(top20_ro_train_text['author'])
y_val_t20ro = l_encoder.transform(top20_ro_val_text['author'])
y_test_t20ro = l_encoder.transform(top20_ro_test_text['author'])

num_authors = len(l_encoder.classes_)
print(f"Numar autori: {num_authors}")
print(f"Autori: {l_encoder.classes_}")



Numar autori: 20
Autori: ['Adrian Dumitru' 'Alexandra Andronie' 'Alexandru Costea'
 'Alexandru Toader' 'Alina Manaila' 'Alina Popescu' 'Andrada Giubega'
 'Andreea Tobias' 'Bogdan Pacurar' 'Daniela Chihaia' 'Diana Nechita'
 'Georgiana Marina' 'Ioana Coman' 'Liviu Cojan' 'Luana Pavaluca'
 'Marco Badea' 'Mihnea Lazar' 'Monica Bonea' 'Robert Kiss'
 'Vlad Mironescu']


In [5]:
# Pas 2: preprocesare text pt bert
# nu putem trimite textul cu mizerii
# curatam si textul si pastram doar asta din datafreames
# convertim in lista pentru siguranta in dataloader
texts_train_t20ro = top20_ro_train_text['text'].apply(clean_text).tolist()
texts_val_t20ro  = top20_ro_val_text['text'].apply(clean_text).tolist()
texts_test_t20ro  = top20_ro_test_text['text'].apply(clean_text).tolist()

In [6]:
# Pas 4 dataset cu sliding window
tokenizer = AutoTokenizer.from_pretrained("dumitrescustefan/bert-base-romanian-cased-v1")

class AuthorDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, 
                 max_length=512, overlap=0.1):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.stride     = int(max_length * (1 - overlap))  # 460
    

    def __len__(self):
        return len(self.texts)
    
    
    def __getitem__(self, idx):
        text  = self.texts[idx]
        label = self.labels[idx]
        
        # tokenizam fara truncation ca sa vedem lungimea reala
        tokens = self.tokenizer(
            text,
            truncation=False,
            add_special_tokens=True,
            return_tensors='pt'
        )
        input_ids = tokens['input_ids'][0]
        
        # daca e scurt - o singura fereastra
        if len(input_ids) <= self.max_length:
            encoded = self.tokenizer(
                text,
                max_length=self.max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            return {
                'input_ids':      encoded['input_ids'].squeeze(0),
                'attention_mask': encoded['attention_mask'].squeeze(0),
                'label':          torch.tensor(label, dtype=torch.long),
                'n_windows':      torch.tensor(1, dtype=torch.long)
            }
        
        # daca e lung - luam prima fereastra pentru training
        # (la inferenta folosim toate ferestrele cu voting)
        input_ids_window = input_ids[:self.max_length]
        attention_mask   = torch.ones(self.max_length, dtype=torch.long)
        
        return {
            'input_ids':      input_ids_window,
            'attention_mask': attention_mask,
            'label':          torch.tensor(label, dtype=torch.long),
            'n_windows':      torch.tensor(
                (len(input_ids) // self.stride) + 1, 
                dtype=torch.long
            )
        }

In [7]:
# Pas 4 cream dataloaderle
BATCH_SIZE = 8  # mic din cauza RAM GPU

train_dataset = AuthorDataset(texts_train_t20ro, y_train_t20ro, tokenizer)
val_dataset   = AuthorDataset(texts_val_t20ro,   y_val_t20ro,   tokenizer)
test_dataset  = AuthorDataset(texts_test_t20ro,  y_test_t20ro,  tokenizer)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    num_workers=2
)
val_loader = DataLoader(
    val_dataset,   
    batch_size=BATCH_SIZE, 
    shuffle=False,
    num_workers=2
)
test_loader = DataLoader(
    test_dataset,  
    batch_size=BATCH_SIZE, 
    shuffle=False,
    num_workers=2
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

Train batches: 6212
Val batches:   168


In [9]:
# Pas 5 Modelul efectiv si forward pass
class BaselineBERT(nn.Module):
    def __init__(self, num_authors, dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(
            "dumitrescustefan/bert-base-romanian-cased-v1"
        )
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_authors)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token
        cls_output = self.dropout(cls_output)
        logits     = self.classifier(cls_output)
        return logits

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device {device}")

Device cuda


In [10]:
model = BaselineBERT(num_authors=num_authors).to(device)

In [11]:
# asta o pun sus
EPOCHS      = 3
LR          = 2e-5
WARMUP_STEPS = len(train_loader) // 10  # 10% din train

In [12]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=len(train_loader) * EPOCHS
)
criterion = nn.CrossEntropyLoss()

In [14]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0

    for i, batch in enumerate(loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        # init gradienti cu 0
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits,labels)

        # asta a fost forward pass
        # acum backprop ca sa setam weights
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if i % 100 == 0:
            print(f"  Batch {i}/{len(loader)} loss: {loss.item():.4f}")
        
    return total_loss / len(loader)

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            
            preds = torch.argmax(logits, dim=-1)
            
            total_loss  += loss.item()
            all_preds   .extend(preds.cpu().numpy())
            all_labels  .extend(labels.cpu().numpy())
    
    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader), f1



In [ ]:
# training loop
best_f1 = 0
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    train_loss = train_epoch(
        model, train_loader, optimizer, scheduler, criterion, device
    )
    val_loss, val_f1 = eval_epoch(model, val_loader, criterion, device)
    
    print(f"Train loss: {train_loss:.4f}")
    print(f"Val loss:   {val_loss:.4f}  |  Val F1: {val_f1:.4f}")
    
    # salvezi cel mai bun model
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'best_model_ro_top20.pt')
        print(f"  Model salvat (F1={val_f1:.4f})")

print(f"\nBest Val F1: {best_f1:.4f}")
# 155 minute F1 la a treia epoca 0.6654 !!!


Epoch 1/3
  Batch 0/6212 loss: 3.2609
  Batch 100/6212 loss: 3.0578
  Batch 200/6212 loss: 2.8245
  Batch 300/6212 loss: 3.2452
  Batch 400/6212 loss: 2.8942
  Batch 500/6212 loss: 2.7714
  Batch 600/6212 loss: 2.4066
  Batch 700/6212 loss: 2.3879
  Batch 800/6212 loss: 2.0306
  Batch 900/6212 loss: 2.1477
  Batch 1000/6212 loss: 1.6149
  Batch 1100/6212 loss: 2.1990
  Batch 1200/6212 loss: 1.6145
  Batch 1300/6212 loss: 1.6249
  Batch 1400/6212 loss: 2.4144
  Batch 1500/6212 loss: 2.5194
  Batch 1600/6212 loss: 2.2293
  Batch 1700/6212 loss: 1.4499
  Batch 1800/6212 loss: 1.9727
  Batch 1900/6212 loss: 1.6158
  Batch 2000/6212 loss: 1.9236
  Batch 2100/6212 loss: 1.6100
  Batch 2200/6212 loss: 2.2963
  Batch 2300/6212 loss: 1.2763
  Batch 2400/6212 loss: 0.9985
  Batch 2500/6212 loss: 2.1979
  Batch 2600/6212 loss: 1.6603
  Batch 2700/6212 loss: 1.7881
  Batch 2800/6212 loss: 1.1329
  Batch 2900/6212 loss: 1.8896
  Batch 3000/6212 loss: 1.8007
  Batch 3100/6212 loss: 2.2357
  Batch 3

In [ ]:
# Pas5 Majority vote pe setul de test
def predict_with_voting(model, text, tokenizer, 
                         max_length=512, overlap=0.1, device='cuda'):
    stride = int(max_length * (1 - overlap))
    model.eval()
    
    tokens = tokenizer(
        text, truncation=False, 
        add_special_tokens=True, return_tensors='pt'
    )
    input_ids = tokens['input_ids'][0]
    
    predictions = []
    
    with torch.no_grad():
        for start in range(0, len(input_ids), stride):
            end      = min(start + max_length, len(input_ids))
            window   = input_ids[start:end]
            
            # padding
            if len(window) < max_length:
                padding = torch.zeros(max_length - len(window), dtype=torch.long)
                window  = torch.cat([window, padding])
            
            attn_mask = (window != 0).long()
            
            logits = model(
                window.unsqueeze(0).to(device),
                attn_mask.unsqueeze(0).to(device)
            )
            pred = torch.argmax(logits, dim=-1).item()
            predictions.append(pred)
            
            if end == len(input_ids):
                break
    
    from collections import Counter
    return Counter(predictions).most_common(1)[0][0]

# evaluezi pe test cu voting
model.load_state_dict(torch.load('best_model_ro_top20.pt'))

all_preds  = []
all_labels = []

for idx in range(len(texts_test_t20ro)):
    pred  = predict_with_voting(model, texts_test_t20ro[idx], tokenizer, device=device)
    all_preds.append(pred)
    all_labels.append(y_test_t20ro[idx])

print(classification_report(
    all_labels, all_preds, 
    target_names=l_encoder.classes_
))